# Case Study: Cyclistic Bike-Share

**How do annual members and casual riders use Cyclistic bikes differently?**<br>
*Google Data Analytics Professional Certificate – Course 8 Capstone*

**Shaine Meister**<br>
**March 28, 2026**

---


## Introduction
This notebook extends the main Cyclistic case study and documents the end-to-end technical workflow used to build an analysis-ready trip dataset for downstream analysis.

Cyclistic is a bike-share program in Chicago with more than 5,800 bicycles and 600 docking stations. The company offers traditional bikes as well as assistive options such as reclining bikes, hand tricycles, and cargo bikes. Casual riders buy single-ride or full-day passes, while annual members purchase yearly memberships.

Cyclistic's finance team has shown that annual members are more profitable than casual riders. To support strategy work in the main case study, this notebook focuses on the backend preparation steps required to ingest raw trip files, validate data quality, clean invalid records, standardize spatial features, enrich trips with hourly weather context, and export a reusable final dataset.


**High-level workflow covered in this notebook**
1. **Prepare** *(Section 2)*
    - Load monthly Divvy trip CSV files for the configured `start_yyyymm` to `end_yyyymm` range.
    - Validate requested files against available S3 objects, download ZIP archives, and extract monthly CSV files.
    - Combine all monthly files into one consolidated dataset.
    - Standardize core data types and run basic schema/readiness checks.

2. **Process** *(Section 3)*
    - Engineer time-based features and apply rule-based data validation checks.
    - Separate records into `clean_data` and `dirty_data` for traceable cleanup.
    - Convert trip coordinates into stable vector-mapped location keys.
    - Merge hourly weather data into trip records using nearest-station matching and audit the merge results.
    - Export the final enriched analytical dataset for downstream analysis and reporting.

This notebook captures the full preparation and processing pipeline that turns raw Cyclistic trip files into a validated, spatially normalized, weather-enriched analytical dataset.

Note: *The following case-study phases are not performed in this notebook: Ask, Analyze, Share, and Act.*

## 2. Prepare
**Data location**  
Public Cyclistic (Divvy) trip data: [Divvy TripData](https://divvy-tripdata.s3.amazonaws.com/index.html "https://divvy-tripdata.s3.amazonaws.com/index.html")<br>
Weather Data: [Open-Meteo](https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv "https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv")<br>
In this section, the notebook builds the raw trip dataset by validating the requested monthly Divvy files against the public S3 bucket, downloading ZIP archives when needed, extracting the monthly CSV files, and combining them into a single DataFrame.

**What this section does**
1. Validate the configured `start_yyyymm` to `end_yyyymm` month range against available Divvy S3 objects.
2. Download missing ZIP archives to a local cache and extract their CSV contents.
3. Concatenate all monthly CSV files into one consolidated dataset.
4. Standardize `started_at` as datetime and sort records chronologically.
5. Run a quick schema sanity check across extracted monthly files.

**How the data is organized** *(sample)* 
<div style="font-size: 10px;">

| `ride_id` | `rideable_type` | `started_at` | `ended_at` | `start_station_name` | `start_station_id` | `end_station_name` | `end_station_id` |
|---|---|---|---|---|---|---|---|
| Unique trip identifier | Bike type used | Trip start timestamp | Trip end timestamp | Origin station name | Origin station ID | Destination station name | Destination station ID |

</div>

The combined trip dataset is expected to contain approximately 5-6 million rows, depending on the selected date range.

**ROCCC verification**  
- Reliable: Collected by Cyclistic's own system.  
- Original: First-party trip data.  
- Comprehensive: Covers every ride in the system.  
- Current: Uses the configured recent month range.  
- Cited: Licensed for analysis (Motivate International).  

**Licensing, privacy, and security**  
Data is public under the Divvy data license. No personally identifiable information is included, so privacy is preserved. Files are cached locally for reproducible reruns of the notebook.

**Prepare outputs produced here**  
- `df`: consolidated raw trip dataset with `started_at` parsed and sorted.  
- `schema_check`: quick cross-file schema QA summary.  

**Data integrity check**  
A quick preview of row structure and column consistency is displayed below before the notebook moves to the next section.

In [32]:
# ================================================================================
# SETUP & CONFIGURATION
# ================================================================================

# Standard library imports
# calendar: monthrange helper; derives the last day of any month for dynamic weather date bounds
# io: in-memory byte/string stream; wraps weather CSV text for pd.read_csv
# re: regular expressions; extracts ZIP filenames from the S3 XML bucket listing
# shutil: high-level file operations for directory removal during cleanup
# zipfile: ZIP archive reading and member extraction for Divvy monthly trip files
# pathlib.Path: object-oriented path handling for local cache directories
# urllib.request: urlopen streams HTTP for S3 XML and weather CSV; urlretrieve downloads ZIPs to disk

# Third-party libraries
# numpy: array operations for boolean dirty-data masks and coordinate grid math
# pandas: core DataFrame library for trip records, weather tables, and grouped output

import calendar
import io
import re
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import numpy as np
import pandas as pd

# ================================================================================
# SECTION 1: GLOBAL PATH CONFIGURATION
# ================================================================================
# Shared path configuration (define once and reuse throughout notebook).
PROJECT_ROOT = Path("/mnt/RepoRetLabs/code/jupyter-notebooks/case-study_bike-share-success")
OUTPUT_DIR = PROJECT_ROOT

# ================================================================================
# SECTION 2: DATE RANGE SETTINGS
# ================================================================================
# Edit these two values to control which months are loaded.
# Format: YYYYMM. Range is inclusive on both ends.
start_yyyymm = "202503"
end_yyyymm = "202602"

# ================================================================================
# SECTION 3: EXPORT FORMAT SETTINGS
# ================================================================================
# Configure export format: binary choice between CSV or ZIP archive.
export_format = 'zip'  # Options: 'csv' or 'zip'

# ================================================================================
# SECTION 4: OUTLIER FLAG SETTINGS
# ================================================================================
# Outlier handling behavior for ride_length_seconds in Process cleanup cell.
# True  -> remove flagged outlier rows from clean_data.
# False -> keep rows and flag outliers in ride_length_outlier column.
remove_flagged_ride_length_outliers = False

In [33]:
# ================================================================================
# PREPARE PHASE: DATA LOADING
# ================================================================================
# Prepare phase: data loading overview
# - Configure S3 source URLs and local ZIP/CSV cache directories.
# - Set target YYYYMM range; build the expected ZIP filename list and verify against S3 availability.
# - Download uncached ZIPs and extract monthly CSVs to disk.
# - Load all monthly CSVs into df; normalize started_at and sort rows chronologically.
# - Fetch Open-Meteo hourly weather CSV once, cache it locally, and parse it into two DataFrames.
# - Print output summary.

# ================================================================================
# SECTION 1: S3 SOURCE URLS & CACHE CONFIGURATION
# ================================================================================
# Base URL for the Divvy public S3 bucket — all ZIP filenames are appended directly to this root.
base_url = "https://divvy-tripdata.s3.amazonaws.com/"
# S3 ListObjectsV2 XML endpoint; more reliable than the HTML index for parsing available file keys.
list_url = base_url + "?list-type=2"

# Workspace root for this project; cache subdirectories are created relative to this path.
data_path = PROJECT_ROOT
zip_dir = data_path / ".divvy_zip"         # Local cache directory for downloaded Divvy ZIP archives
extract_dir = data_path / ".divvy_csv"     # Local cache directory for extracted Divvy monthly CSV files
weather_dir = data_path / ".weather_csv"   # Local cache directory for Open-Meteo weather CSV responses
zip_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)
weather_dir.mkdir(parents=True, exist_ok=True)

# ================================================================================
# SECTION 2: YYYYMM RANGE SETUP & S3 AVAILABILITY
# ================================================================================
# iter_yyyymm: generator that yields every YYYYMM string in a closed [start, end] interval.
# Increments the month counter and rolls December (12) over to January (1) of the next year
# so that multi-year ranges are handled correctly without relying on dateutil or pandas.
def iter_yyyymm(start_yyyymm, end_yyyymm):
    """Yield consecutive YYYYMM strings from start to end (inclusive), handling December-to-January rollover."""
    y, m = int(start_yyyymm[:4]), int(start_yyyymm[4:6])
    end_y, end_m = int(end_yyyymm[:4]), int(end_yyyymm[4:6])

    while (y < end_y) or (y == end_y and m <= end_m):
        yield f"{y:04d}{m:02d}"
        m += 1
        if m > 12:   # December rolls over to January of the following year
            y += 1
            m = 1

# Build the full list of expected ZIP filenames for every month in the requested range.
desired_zip_files = [f"{yyyymm}-divvy-tripdata.zip" for yyyymm in iter_yyyymm(start_yyyymm, end_yyyymm)]

# S3 availability check
# Fetch the bucket's XML object listing and extract all valid YYYYMM ZIP keys using the file pattern.
# Each available file appears as <Key>YYYYMM-divvy-tripdata.zip</Key> in the XML response body.
with urlopen(list_url) as response:
    listing_xml = response.read().decode("utf-8", errors="ignore")

available_zip_files = set(re.findall(r"<Key>(20\d{4}-divvy-tripdata\.zip)</Key>", listing_xml))

# Intersection: only process months that fall within the requested range AND exist in S3.
zip_files = [z for z in desired_zip_files if z in available_zip_files]
# Any month in the requested range but absent from S3 is recorded separately and reported below.
missing_zip_files = [z for z in desired_zip_files if z not in available_zip_files]

if not zip_files:
    raise ValueError(
        f"No Divvy monthly ZIP files found in S3 for range {start_yyyymm} to {end_yyyymm}."
    )

print(f"Requested months: {len(desired_zip_files)}")
print(f"Available ZIP files to process: {len(zip_files)}")
if missing_zip_files:
    print(f"Missing months in S3 (skipped): {len(missing_zip_files)}")
    print("Examples:", missing_zip_files[:5])

# ================================================================================
# SECTION 3: ZIP DOWNLOAD & CSV EXTRACTION
# ================================================================================
extracted_csv_paths = []
for zip_name in zip_files:
    zip_path = zip_dir / zip_name
    zip_url = base_url + zip_name

    # Skip download if the ZIP is already cached locally to avoid redundant network requests.
    if not zip_path.exists():
        print(f"Downloading: {zip_name}")
        urlretrieve(zip_url, zip_path)

    # Derive the expected flat CSV filename from the ZIP name (e.g. 202503-divvy-tripdata.csv).
    expected_csv_name = zip_name.replace(".zip", ".csv")
    expected_csv_path = extract_dir / expected_csv_name

    # Skip extraction if the CSV was already unpacked in a previous run.
    if not expected_csv_path.exists():
        with zipfile.ZipFile(zip_path, "r") as zf:
            csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
            if not csv_members:
                raise ValueError(f"No CSV found in ZIP: {zip_name}")

            member = csv_members[0]
            zf.extract(member, path=extract_dir)

            # Some ZIPs embed the CSV inside a subdirectory (e.g. "subdir/YYYYMM-*.csv").
            # Rename the extracted file to a flat path in extract_dir for consistent access.
            extracted_member_path = extract_dir / member
            if extracted_member_path != expected_csv_path:
                extracted_member_path.replace(expected_csv_path)

    extracted_csv_paths.append(expected_csv_path)

# Plain filename list retained for compatibility with downstream notebook references.
csv_files = [p.name for p in extracted_csv_paths]

# ================================================================================
# SECTION 4: CSV LOAD & CONCATENATION
# ================================================================================
df_list = [pd.read_csv(p) for p in extracted_csv_paths]
df = pd.concat(df_list, ignore_index=True)

# Parse started_at to datetime and sort rows chronologically; required for time-based
# feature engineering and consistent sliding-window operations in the Process section.
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df = df.sort_values("started_at").reset_index(drop=True)

# ================================================================================
# SECTION 5: WEATHER DATA IMPORT (OPEN-METEO)
# ================================================================================
# Cache the weather CSV locally so repeated notebook runs reuse the prior download,
# matching the same cache-first behavior used for Divvy trip files.
# Date bounds are derived automatically from start_yyyymm / end_yyyymm:
#   weather_start_date = last day of the month immediately before start_yyyymm
#   weather_end_date   = last day of end_yyyymm

start_y, start_m = int(start_yyyymm[:4]), int(start_yyyymm[4:6])
prev_m = start_m - 1
prev_y = start_y
if prev_m == 0:   # January rolls back to December of the prior year
    prev_m = 12
    prev_y -= 1
weather_start_date = f"{prev_y:04d}-{prev_m:02d}-{calendar.monthrange(prev_y, prev_m)[1]:02d}"

end_y, end_m = int(end_yyyymm[:4]), int(end_yyyymm[4:6])
weather_end_date = f"{end_y:04d}-{end_m:02d}-{calendar.monthrange(end_y, end_m)[1]:02d}"

weather_file = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude=41.65,42.10&longitude=-87.85,-87.40"
    f"&start_date={weather_start_date}&end_date={weather_end_date}"
    f"&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,"
    f"wind_speed_10m,wind_direction_10m,cloud_cover"
    f"&timezone=America/Chicago&format=csv"
)
weather_cache_path = weather_dir / f"open_meteo_{weather_start_date}_{weather_end_date}.csv"

if not weather_cache_path.exists():
    print(f"Downloading weather cache: {weather_cache_path.name}")
    urlretrieve(weather_file, weather_cache_path)

# Read the cached weather response and split it into individual lines so the
# dual-table boundary can be detected before passing each slice to pd.read_csv.
lines = weather_cache_path.read_text(encoding="utf-8", errors="ignore").splitlines()

# Locate the index of the second 'location_id' header row (index 0 is always Table 1's header).
# Everything before split_idx belongs to Table 1; from split_idx onward is Table 2.
split_idx = next(
    i for i in range(1, len(lines))
    if lines[i].strip().startswith("location_id")
)

# Rejoin each slice with newlines so pd.read_csv receives a properly row-delimited CSV string.
weather_locations_csv = "\n".join(lines[:split_idx])
weather_observations_csv = "\n".join(lines[split_idx:])

# Parse each table slice into its own DataFrame for independent use in downstream analysis.
weather_locations_df = pd.read_csv(io.StringIO(weather_locations_csv))
weather_df = pd.read_csv(io.StringIO(weather_observations_csv))

# ================================================================================
# SECTION 6: PREP OUTPUT SUMMARY
# ================================================================================
print(f"CSV files loaded: {len(extracted_csv_paths)}")
print(f"Rows loaded: {len(df):,}")
display(df.head())

print("\nWeather locations:")
display(weather_locations_df.head())

print("Weather observations:")
display(weather_df.head())


Requested months: 12
Available ZIP files to process: 12
Downloading: 202506-divvy-tripdata.zip
Downloading: 202507-divvy-tripdata.zip
Downloading: 202508-divvy-tripdata.zip
Downloading: 202509-divvy-tripdata.zip
Downloading: 202510-divvy-tripdata.zip
Downloading: 202511-divvy-tripdata.zip
Downloading: 202512-divvy-tripdata.zip
Downloading: 202601-divvy-tripdata.zip
Downloading: 202602-divvy-tripdata.zip
CSV files loaded: 12
Rows loaded: 5,601,662


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,5FBF1BFE4C72756F,classic_bike,2025-02-28 08:06:36.009,2025-03-01 09:06:25.127,Orleans St & Hubbard St,636,NaN,NaN,41.890028,-87.636618,NaN,NaN,member
1,FDA49BB617F644B4,classic_bike,2025-02-28 12:29:55.028,2025-03-01 13:29:33.874,Sheridan Rd & Montrose Ave,TA1307000107,NaN,NaN,41.961670,-87.654640,NaN,NaN,casual
2,8B20E2B0D8603957,classic_bike,2025-02-28 13:53:38.438,2025-03-01 14:53:31.545,DuSable Lake Shore Dr & Monroe St,13300,NaN,NaN,41.880958,-87.616743,NaN,NaN,casual
3,B7FC6F4C1F021136,classic_bike,2025-02-28 13:58:06.709,2025-03-01 14:58:00.064,Broadway & Granville Ave,15571,NaN,NaN,41.994780,-87.660285,NaN,NaN,member
4,F837FF70199A73B3,classic_bike,2025-02-28 14:02:54.583,2025-03-01 15:02:34.239,Phillips Ave & 83rd St,582,NaN,NaN,41.744531,-87.565060,NaN,NaN,casual



Weather locations:


,location_id,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
0,0,41.65202,-87.78903,215.0,-18000,America/Chicago,GMT-5
1,1,42.07381,-87.37610,174.0,-18000,America/Chicago,GMT-5


Weather observations:


,location_id,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),rain (mm),snowfall (cm),wind_speed_10m (km/h),wind_direction_10m (°),cloud_cover (%)
0,0,2025-02-28T00:00,1.1,76,0.0,0.0,0.0,9.0,244,89
1,0,2025-02-28T01:00,0.8,77,0.0,0.0,0.0,10.5,235,97
2,0,2025-02-28T02:00,0.6,78,0.0,0.0,0.0,10.0,231,53
3,0,2025-02-28T03:00,0.4,79,0.0,0.0,0.0,12.7,230,0
4,0,2025-02-28T04:00,0.4,79,0.0,0.0,0.0,12.9,221,26


In [34]:
# ================================================================================
# PREPARE PHASE QA: SCHEMA SANITY CHECK
# ================================================================================
# Optional QA: quick schema sanity check across extracted monthly CSV files.
# Purpose: catch obvious structural drift before concatenation by comparing
# column counts and a small dtype sample from each file.
# Limitations: this does not validate full column names, column order, or
# late-file type issues because it reads only the first 5 rows per file.
schema_rows = []
for csv_path in extracted_csv_paths:
    tmp = pd.read_csv(csv_path, nrows=5)
    schema_rows.append({
        "file": csv_path.name,
        "columns": len(tmp.columns),           # Expected to match across all months
        "sample_dtypes": ", ".join(tmp.dtypes.astype(str).head(5).tolist()),  # Quick sample, not full-schema validation
    })

schema_check = pd.DataFrame(schema_rows).sort_values("file").reset_index(drop=True)
display(schema_check.head(12))

# A single unique column-count value is a useful early signal, not a complete schema guarantee.
print("Unique column counts across files:", sorted(schema_check["columns"].unique()))
print("Quick schema sanity check complete.")


,file,columns,sample_dtypes
0,202503-divvy-tripdata.csv,13,"object, object, object, object, float64"
1,202504-divvy-tripdata.csv,13,"object, object, object, object, object"
2,202505-divvy-tripdata.csv,13,"object, object, object, object, object"
3,202506-divvy-tripdata.csv,13,"object, object, object, object, object"
4,202507-divvy-tripdata.csv,13,"object, object, object, object, object"
5,202508-divvy-tripdata.csv,13,"object, object, object, object, float64"
6,202509-divvy-tripdata.csv,13,"object, object, object, object, float64"
7,202510-divvy-tripdata.csv,13,"object, object, object, object, object"
8,202511-divvy-tripdata.csv,13,"object, object, object, object, float64"
9,202512-divvy-tripdata.csv,13,"object, object, object, object, object"


Unique column counts across files: [np.int64(13)]
Quick schema sanity check complete.


## 3. Process
**Tools of choice**<br>
Python with pandas and numpy - scalable, reproducible, and suitable for large trip data and downstream enrichment workflows.

This section covers the full processing pipeline applied after the raw trip files are prepared. It starts with Divvy trip validation and feature engineering, removes rule-flagged bad records, converts raw coordinates into stable vector-mapped location keys, enriches trips with nearest-station hourly weather data, and finishes by exporting the final enriched dataset for downstream analysis.

**Workflow Summary:**
1. **Divvy Data Processing**
   - Standardize datetime fields, calculate `ride_length_seconds`, round timestamps, and derive reusable time features.
   - Build reusable validation masks and dirty-data rules for chronology, missing values, and duplicate records.
   - Generate QA tables, A/B comparisons, and reconciliation summaries.

2. **Bad Data Clean Up and Drop**
   - Combine all rule flags into one dirty-row mask.
   - Split records into `dirty_data` and `clean_data`, remove invalid rows from the working dataset, and drop no-longer-needed columns.

3. **Coordinate Vector Mapping**
   - Build a geographic bounding box from trip coordinates.
   - Map start and end latitude/longitude values into stable grid-based vector keys for downstream grouping and comparison.

4. **Weather Data Processing and Merge**
   - Match each trip start grid point to the nearest weather station.
   - Merge hourly weather observations into trip records and produce merge-audit diagnostics.

5. **Export Final Enriched Data**
   - Export the final processed dataset as CSV, with optional Excel output.

**Primary outputs from this section**
- `clean_data` and `dirty_data` for traceable record separation.  
- Vector-mapped trip fields for location-based grouping.  
- `df_weather_merge` as the final weather-enriched analytical dataset.  
- QA and audit tables that document validation and merge quality.

### Divvy Data Processing - Script flow (order of operation)
This markdown reflects the sectioned comments inside the Python script.

1. **Section 1 - Configuration & setup**
   - Defines station-name cleanup controls: `STATION_NAME_REMOVABLE_CHARS`, `STATION_NAME_SAMPLE_LIMIT_PER_CHAR`, and regex pattern build logic.

2. **Section 2 - Datetime parsing & month range filtering**
   - Parses `started_at` and `ended_at` to datetime.
   - Applies strict month-window filtering where both timestamps must be within `[start_yyyymm, end_yyyymm]`.

3. **Section 3 - Feature engineering (duration + timestamp rounding)**
   - Computes `ride_length_seconds` and rounds to configurable precision (current: 10s).
   - Rounds timestamps to a grouping interval (current: 300s / 5 minutes).

4. **Section 4 - Station-name cleanup**
   - Normalizes text (trim/collapse spaces), removes configured characters, and converts empty strings to null.
   - Builds cleanup QA outputs: row-change summary, removed-character counts, and before/after examples.

5. **Section 5 - Time dimensions**
   - Derives `dt_start_hour` and `day_of_week` for downstream analysis.

6. **Section 6 - Validation framework (functions + purpose)**
   - **6a `build_rule_masks(df)`**: builds reusable boolean masks for chronology validation and missing location fields.
   - **6b `build_dirty_rules(df, masks)`**: defines rule metadata (mask, expected value, why_dirty) for each dirty-data condition.
   - **6c `build_ab_cols_map()`**: maps each rule to the columns shown in A/B dirty-vs-clean QA examples.
   - **6d `summarize_dirty_rules(df, dirty_rules)`**: computes per-rule counts/percentages plus union and overlap diagnostics.
   - **6e `display_ab_showcase(df, dirty_rules, ab_cols_map)`**: prints and displays side-by-side A/B samples for triggered rules.
   - **6f `build_location_cross_check(df, masks)`**: creates required-field null-count and null-rate QA table for start/end location columns.
   - **6g `build_validation_summary(df, masks, case_flags, any_dirty_count)`**: returns compact totals and union-consistency checks.
   - **6h Execute validation workflow**: runs masks -> rules -> summaries -> cross-checks and materializes `case_summary`, `rule_count_check`, `chronology_cross_check`, `location_cross_check`, and `validation_summary`.

7. **Section 7 - QA output display**
   - Displays A/B examples, dirty overview, and validation summary.

8. **Section 8 - Data partition & reconciliation audit**
   - Materializes `dirty_data` and `clean_data` from `combined_dirty_mask`.
   - Creates `cleaning_audit` with row-level reconciliation checks.

9. **Section 9 - Output summary & variable inventory**
   - Documents output objects by category: validation framework, summaries, quality checks, station cleanup outputs, partitions, and audit fields.

**Outputs from this cell**
- Validation framework: `masks`, `dirty_rules`, `case_flags`, `ab_cols_map`
- Validation summaries: `case_summary`, `validation_summary`, `rule_count_check`
- Quality checks: `any_dirty_count`, `multi_rule_count`, `chronology_cross_check`, `location_cross_check`
- Station-name cleanup: `station_name_cleanup_summary`, `station_name_removed_char_summary`, `station_name_removed_char_examples`
- Data partitions: `combined_dirty_mask`, `dirty_data`, `clean_data`
- Audit fields: `cleaning_audit`, `union_recalc`, `union_check_ok`

In [35]:
from collections import Counter, defaultdict

# ================================================================================
# SECTION 1: CONFIGURATION & SETUP
# ================================================================================
# Station-name cleanup configuration.
# Keep this list short and explicit so character-removal rules are easy to change later.
STATION_NAME_REMOVABLE_CHARS = ['*']
STATION_NAME_SAMPLE_LIMIT_PER_CHAR = 3
STATION_NAME_REMOVABLE_CHAR_PATTERN = (
    '[' + re.escape(''.join(STATION_NAME_REMOVABLE_CHARS)) + ']'
    if STATION_NAME_REMOVABLE_CHARS
    else r'(?!)'  # Regex that matches nothing when no removable chars are configured.
)

# ================================================================================
# SECTION 2: DATETIME PARSING & MONTH RANGE FILTERING
# ================================================================================
# Parse and normalize datetime fields used throughout validation and feature engineering.
df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')

# Strict month filter: keep only rows where BOTH started_at and ended_at
# fall within the configured inclusive [start_yyyymm, end_yyyymm] month window.
rows_before_month_filter = len(df)
started_yyyymm = df['started_at'].dt.strftime('%Y%m')
ended_yyyymm = df['ended_at'].dt.strftime('%Y%m')
month_window_mask = (
    df['started_at'].notna()
    & df['ended_at'].notna()
    & started_yyyymm.between(start_yyyymm, end_yyyymm)
    & ended_yyyymm.between(start_yyyymm, end_yyyymm)
)
df = df.loc[month_window_mask].copy()
rows_after_month_filter = len(df)
print(
    f"Month-range filter applied ({start_yyyymm} to {end_yyyymm}): "
    f"kept {rows_after_month_filter:,} / {rows_before_month_filter:,} rows "
    f"(removed {rows_before_month_filter - rows_after_month_filter:,})."
)

# ================================================================================
# SECTION 3: FEATURE ENGINEERING — RIDE DURATION & TIMESTAMP ROUNDING
# ================================================================================

# Compute trip duration in seconds, then round to a configurable precision.
# Lower values keep finer precision; higher values simplify durations.
ride_length_round_to_seconds = 10
ride_length_seconds = (df['ended_at'] - df['started_at']).dt.total_seconds()
df['ride_length_seconds'] = ((ride_length_seconds / ride_length_round_to_seconds).round() * ride_length_round_to_seconds).astype('Int64')

# Round timestamps to a configurable interval for consistent temporal grouping.
# Example values: 10s, 15s, 30s, 60s, 300s.
round_time_to_seconds = 300
round_freq = f"{round_time_to_seconds}s"
df['started_at'] = df['started_at'].dt.round(round_freq)
df['ended_at'] = df['ended_at'].dt.round(round_freq)

# ================================================================================
# SECTION 4: DATA CLEANING — STATION NAME TEXT NORMALIZATION
# ================================================================================
# Clean station-name text before validation rules are evaluated.
# Rules:
# - trim edge whitespace
# - collapse repeated internal spaces
# - remove only configured removable characters (currently '*')
# - convert empty results to null

# Normalize station-name strings using whitespace cleanup + configurable removable characters.
def normalize_station_name(series):
    cleaned = series.astype('string')
    cleaned = cleaned.str.strip()
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True)
    cleaned = cleaned.str.replace(STATION_NAME_REMOVABLE_CHAR_PATTERN, "", regex=True)
    cleaned = cleaned.str.strip()
    cleaned = cleaned.mask(cleaned.eq(''), pd.NA)
    return cleaned


# Return only configured removable characters found in the input value.
def get_removed_chars_for_value(original):
    trimmed = original.strip()
    whitespace_normalized = re.sub(r'\s+', ' ', trimmed)
    return re.findall(STATION_NAME_REMOVABLE_CHAR_PATTERN, whitespace_normalized)


# Build per-character removal counts and before/after samples for one station-name field.
def collect_removed_station_name_chars_and_samples(original_series, cleaned_series, field_name, sample_limit_per_char=3):
    removed_counts = Counter()
    removed_samples = defaultdict(list)

    for original_val, cleaned_val in zip(original_series, cleaned_series):
        if pd.isna(original_val):
            continue

        original = str(original_val)
        cleaned = '<NA>' if pd.isna(cleaned_val) else str(cleaned_val)
        removed_chars = get_removed_chars_for_value(original)
        if not removed_chars:
            continue

        removed_counts.update(removed_chars)
        for char in set(removed_chars):
            if len(removed_samples[char]) < sample_limit_per_char:
                removed_samples[char].append({
                    'field': field_name,
                    'sample_before': original,
                    'sample_after': cleaned,
                })

    return removed_counts, removed_samples


# Convert whitespace and control characters to readable labels for display tables.
def label_removed_character(char):
    labels = {
        ' ': '<space>',
        '\t': '<tab>',
        '\n': '<newline>',
        '\r': '<carriage_return>',
    }
    return labels.get(char, char)


# Apply station-name cleanup to both start_station_name and end_station_name fields.
station_name_removed_char_counts = Counter()
station_name_removed_char_samples = defaultdict(list)
station_name_cleanup_rows = []
for station_col in ['start_station_name', 'end_station_name']:
    if station_col in df.columns:
        original_series = df[station_col].copy()
        cleaned_series = normalize_station_name(original_series)

        char_counts, char_samples = collect_removed_station_name_chars_and_samples(
            original_series=original_series,
            cleaned_series=cleaned_series,
            field_name=station_col,
            sample_limit_per_char=STATION_NAME_SAMPLE_LIMIT_PER_CHAR,
        )
        station_name_removed_char_counts.update(char_counts)
        for char, sample_list in char_samples.items():
            remaining_slots = STATION_NAME_SAMPLE_LIMIT_PER_CHAR - len(station_name_removed_char_samples[char])
            if remaining_slots > 0:
                station_name_removed_char_samples[char].extend(sample_list[:remaining_slots])

        station_name_cleanup_rows.append({
            'field': station_col,
            'rows_changed': int(original_series.astype('string').fillna(pd.NA).ne(cleaned_series).fillna(False).sum()),
            'rows_null_after_cleanup': int(cleaned_series.isna().sum()),
        })
        df[station_col] = cleaned_series

station_name_cleanup_summary = pd.DataFrame(station_name_cleanup_rows)

station_name_removed_char_rows = [
    {
        'removed_character': label_removed_character(char),
        'removed_count': count,
        'unicode_codepoint': f"U+{ord(char):04X}",
    }
    for char, count in station_name_removed_char_counts.items()
]
if station_name_removed_char_rows:
    station_name_removed_char_summary = pd.DataFrame(station_name_removed_char_rows).sort_values(
        ['removed_count', 'removed_character'], ascending=[False, True]
    ).reset_index(drop=True)
else:
    station_name_removed_char_summary = pd.DataFrame(
        columns=['removed_character', 'removed_count', 'unicode_codepoint']
    )

station_name_removed_char_example_rows = [
    {
        'removed_character': label_removed_character(char),
        'unicode_codepoint': f"U+{ord(char):04X}",
        'field': sample['field'],
        'sample_before': sample['sample_before'],
        'sample_after': sample['sample_after'],
    }
    for char, samples in station_name_removed_char_samples.items()
    for sample in samples
]
if station_name_removed_char_example_rows:
    station_name_removed_char_examples = pd.DataFrame(station_name_removed_char_example_rows).sort_values(
        ['removed_character', 'field', 'sample_before']
    ).reset_index(drop=True)
else:
    station_name_removed_char_examples = pd.DataFrame(
        columns=['removed_character', 'unicode_codepoint', 'field', 'sample_before', 'sample_after']
    )

print("\nStation-name cleanup summary:")
display(station_name_cleanup_summary)
if station_name_removed_char_summary.empty:
    print("No configured station-name characters were removed by cleanup rules.")
else:
    print("Removed configured characters by unique character:")
    display(station_name_removed_char_summary)

if station_name_removed_char_examples.empty:
    print("No before/after samples available for removed configured characters.")
else:
    print("Before/after samples by removed character (up to configured sample limit per character):")
    display(station_name_removed_char_examples)

# ================================================================================
# SECTION 5: FEATURE ENGINEERING — TIME DIMENSIONS
# ================================================================================
# Derive reusable time dimensions for downstream analysis.
# Hour bucket from started_at for time-of-day analysis.
df['dt_start_hour'] = df['started_at'].dt.floor('h')

# Day-of-week index where Monday=0 and Sunday=6.
df['day_of_week'] = df['started_at'].dt.dayofweek

# ================================================================================
# SECTION 6: VALIDATION FRAMEWORK — BUILDING MASKS, RULES & BUSINESS LOGIC
# ================================================================================
# Purpose: Build a reusable validation framework with modular components.
# Flow: 6a Masks → 6b Rules → 6c A/B map → 6d Rule summary → 6e A/B display → 6f Location cross-check → 6g Validation summary → 6h Execute

# -------------------------------------------------------------------------------
# SECTION 6a: BUILD RULE MASKS
# -------------------------------------------------------------------------------
# Build reusable low-level validation masks for dirty-data rules.
def build_rule_masks(df):
    negative_ride_length_mask = df['ride_length_seconds'] < 0
    end_before_start_mask = df['ended_at'] < df['started_at']
    duration_validation_mask = negative_ride_length_mask | end_before_start_mask

    location_required_cols = [
        'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
        'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
    ]
    location_fields_missing_mask = df[location_required_cols].isna().any(axis=1)

    return {
        'negative_ride_length_mask': negative_ride_length_mask,
        'end_before_start_mask': end_before_start_mask,
        'duration_validation_mask': duration_validation_mask,
        'location_required_cols': location_required_cols,
        'location_fields_missing_mask': location_fields_missing_mask,
    }


# -------------------------------------------------------------------------------
# SECTION 6b: BUILD DIRTY-DATA RULES
# -------------------------------------------------------------------------------
# Define primary dirty-data rules from reusable masks and business expectations.
def build_dirty_rules(df, masks):
    return {
        'Invalid trip duration chronology': {
            'mask': masks['duration_validation_mask'],
            'expected': 'ride_length_seconds > 0 and ended_at >= started_at',
            'why_dirty': 'Negative duration or end before start.'
        },
        'Zero ride length': {
            'mask': df['ride_length_seconds'] == 0,
            'expected': 'ride_length_seconds > 0',
            'why_dirty': 'Zero-duration trip.'
        },
        'Missing rider type': {
            'mask': df['member_casual'].isna(),
            'expected': "member_casual in {'member','casual'}",
            'why_dirty': 'Rider segment missing.'
        },
        'Missing trip location context (start OR end)': {
            'mask': masks['location_fields_missing_mask'],
            'expected': 'All start/end location fields are populated (no nulls)',
            'why_dirty': 'At least one required location field is missing.',
        },
        'Duplicate ride_id': {
            'mask': df.duplicated(subset='ride_id', keep=False),
            'expected': 'ride_id unique per trip',
            'why_dirty': 'Duplicate trip IDs.'
        },
    }


# -------------------------------------------------------------------------------
# SECTION 6c: BUILD A/B COLUMN MAP
# -------------------------------------------------------------------------------
# Map each dirty-data rule to columns shown in A/B QA examples.
def build_ab_cols_map():
    return {
        'Invalid trip duration chronology': ['ride_id', 'started_at', 'ended_at', 'ride_length_seconds'],
        'Zero ride length': ['ride_id', 'started_at', 'ended_at', 'ride_length_seconds'],
        'Missing rider type': ['ride_id', 'member_casual'],
        'Missing trip location context (start OR end)': [
            'ride_id',
            'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
            'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
        ],
        'Duplicate ride_id': ['ride_id', 'started_at', 'ended_at', 'rideable_type', 'member_casual'],
    }


# -------------------------------------------------------------------------------
# SECTION 6d: SUMMARIZE DIRTY RULES
# -------------------------------------------------------------------------------
# Summarize rule-level counts and compute union/overlap diagnostics.
def summarize_dirty_rules(df, dirty_rules):
    total_rows = len(df)
    case_rows = []

    for case_name, rule in dirty_rules.items():
        count = len(df[rule['mask']])
        case_rows.append({
            'dirty_case': case_name,
            'count': count,
            'pct_of_rows': round((count / total_rows) * 100, 4),
            'expected_value': rule['expected'],
            'why_considered_dirty': rule['why_dirty'],
        })

    case_flags = pd.DataFrame({name: rule['mask'].to_numpy() for name, rule in dirty_rules.items()}, index=df.index)

    any_dirty_count = len(df[case_flags.any(axis=1)])
    multi_rule_count = len(df[case_flags.sum(axis=1) > 1])

    case_rows.append({
        'dirty_case': 'ANY dirty row (union of all rules)',
        'count': any_dirty_count,
        'pct_of_rows': round((any_dirty_count / total_rows) * 100, 4),
        'expected_value': 'At least one rule violated',
        'why_considered_dirty': 'Union count.'
    })

    case_rows.append({
        'dirty_case': 'Rows flagged by >=2 rules (overlap)',
        'count': multi_rule_count,
        'pct_of_rows': round((multi_rule_count / total_rows) * 100, 4),
        'expected_value': 'Prefer one issue per row',
        'why_considered_dirty': 'Overlap count.'
    })

    case_summary = pd.DataFrame(case_rows).sort_values('count', ascending=False)
    return case_summary, case_flags, any_dirty_count, multi_rule_count


# -------------------------------------------------------------------------------
# SECTION 6e: DISPLAY A/B QA SHOWCASE
# -------------------------------------------------------------------------------
# Display A/B examples for each triggered dirty-data rule.
def display_ab_showcase(df, dirty_rules, ab_cols_map):
    hidden_display_cols = {'ride_id', 'start_station_id', 'end_station_id'}

    for case_name, rule in dirty_rules.items():
        mask = rule['mask']
        if not mask.any():
            continue

        cols = ab_cols_map.get(case_name, ['ride_id'])
        display_cols = [col for col in cols if col not in hidden_display_cols] or cols

        print(f"\n=== {case_name} ===")
        print(f"Total flagged rows: {len(df[mask]):,}")
        print(f"Total non-flagged rows: {len(df[~mask]):,}")
        print(f"Expected: {rule['expected']}")
        print(f"Why dirty: {rule['why_dirty']}")

        a_dirty = df.loc[mask, display_cols].head(3).copy()
        a_dirty.insert(0, 'A/B', 'A (dirty)')

        b_clean = df.loc[~mask, display_cols].head(3).copy()
        b_clean.insert(0, 'A/B', 'B (clean)')

        display(pd.concat([a_dirty, b_clean], ignore_index=True))


# -------------------------------------------------------------------------------
# SECTION 6f: BUILD LOCATION CROSS-CHECK
# -------------------------------------------------------------------------------
# Build a null-count QA table for required start/end location columns.
def build_location_cross_check(df, masks):
    location_required_cols = masks['location_required_cols']
    total_rows = len(df)

    location_cross_check = pd.DataFrame({
        'field': location_required_cols,
        'missing_rows': [int(df[col].isna().sum()) for col in location_required_cols],
    })
    location_cross_check['missing_pct'] = (location_cross_check['missing_rows'] / total_rows * 100).round(4)

    return location_cross_check.sort_values('missing_rows', ascending=False)


# -------------------------------------------------------------------------------
# SECTION 6g: BUILD VALIDATION SUMMARY
# -------------------------------------------------------------------------------
# Build a compact validation summary with totals and union-consistency checks.
def build_validation_summary(df, masks, case_flags, any_dirty_count):
    total_rows = len(df)
    missing_location_rows = int(masks['location_fields_missing_mask'].sum())
    complete_location_rows = int((~masks['location_fields_missing_mask']).sum())
    union_recalc = int(case_flags.any(axis=1).sum())

    return pd.DataFrame([
        {'check': 'Total rows', 'value': total_rows},
        {'check': 'Dirty rows (union)', 'value': any_dirty_count},
        {'check': 'Rows with complete location fields', 'value': complete_location_rows},
        {'check': 'Rows with missing location fields', 'value': missing_location_rows},
        {'check': 'Union count consistent', 'value': union_recalc == any_dirty_count},
    ])


# ================================================================================
# SECTION 6h: EXECUTE VALIDATION WORKFLOW
# ================================================================================
# Execute the validation workflow in order: masks -> rules -> summaries -> cross-checks.
masks = build_rule_masks(df)
dirty_rules = build_dirty_rules(df, masks)
ab_cols_map = build_ab_cols_map()

case_summary, case_flags, any_dirty_count, multi_rule_count = summarize_dirty_rules(df, dirty_rules)
rule_count_check = pd.DataFrame([
    {'rule': name, 'count': len(df[rule['mask']])}
    for name, rule in dirty_rules.items()
])

chronology_cross_check = pd.DataFrame([
    {'metric': 'Negative ride length rows', 'value': int(masks['negative_ride_length_mask'].sum())},
    {'metric': 'End time before start time rows', 'value': int(masks['end_before_start_mask'].sum())},
    {'metric': 'Combined chronology-invalid rows', 'value': int(masks['duration_validation_mask'].sum())},
])

location_cross_check = build_location_cross_check(df, masks)
validation_summary = build_validation_summary(df, masks, case_flags, any_dirty_count)

# ================================================================================
# SECTION 7: QA OUTPUT & RESULTS DISPLAY
# ================================================================================
# Display QA outputs used to validate rule behavior and aggregate results.
display_ab_showcase(df, dirty_rules, ab_cols_map)

print('\n=== Dirty Data Overview ===')
display(case_summary)

print('\n=== Validation Summary ===')
display(validation_summary)


# ================================================================================
# SECTION 8: DATA PARTITION & RECONCILIATION AUDIT
# ================================================================================
# Materialize dirty/clean row partitions and build a one-table reconciliation audit.
combined_dirty_mask = case_flags.any(axis=1)

dirty_data = df.loc[combined_dirty_mask].copy()
clean_data = df.loc[~combined_dirty_mask].copy()

original_shape = df.shape
dirty_count = len(dirty_data)
clean_count = len(clean_data)
union_recalc = int(combined_dirty_mask.sum())
union_check_ok = union_recalc == any_dirty_count

cleaning_audit = pd.DataFrame([
    {"metric": "original_rows", "value": original_shape[0]},
    {"metric": "dirty_rows", "value": dirty_count},
    {"metric": "clean_rows", "value": clean_count},
    {"metric": "union_recalc", "value": union_recalc},
    {"metric": "union_matches_any_dirty_count", "value": union_check_ok},
])

# print("\n=== Cleaning Audit ===")
# display(cleaning_audit)

# ================================================================================
# SECTION 9: OUTPUT SUMMARY & VARIABLE INVENTORY
# ================================================================================
# All variables created by this cell are grouped by category below for easy reference.

# === VALIDATION FRAMEWORK ===
# - masks (dict): boolean masks for validation checks
# - dirty_rules (dict): rule definitions with mask, expected value, reason
# - case_flags (DataFrame): rule-flag matrix (one column per rule)
# - ab_cols_map (dict): columns for A/B dirty-vs-clean samples per rule

# === VALIDATION SUMMARIES ===
# - case_summary (DataFrame): per-rule counts, percentages, union, overlap
# - validation_summary (DataFrame): compact totals and completeness checks
# - rule_count_check (DataFrame): per-rule counts for reconciliation

# === DATA QUALITY CHECKS ===
# - any_dirty_count (int): rows flagged by at least one rule
# - multi_rule_count (int): rows flagged by multiple rules
# - chronology_cross_check (DataFrame): chronology validation QA
# - location_cross_check (DataFrame): location field null profile

# === STATION NAME CLEANUP ===
# - station_name_cleanup_summary (DataFrame): per-field cleanup row counts
# - station_name_removed_char_summary (DataFrame): character removal aggregates
# - station_name_removed_char_examples (DataFrame): before/after samples

# === DERIVED FIELDS ===
# - dt_start_hour (Series): started_at truncated to hour

# === DATA PARTITIONS (Below) ===
# - combined_dirty_mask (Series): union mask from all rules
# - dirty_data (DataFrame): rows flagged by any rule
# - clean_data (DataFrame): rows passing all rules
# - non_outlier_clean_data (DataFrame): clean data excluding ride-length outliers

# === AUDIT FIELDS ===
# - original_shape (tuple): shape before filtering
# - dirty_count (int): flagged row count
# - clean_count (int): passing row count
# - union_recalc (int): recomputed union count
# - union_check_ok (bool): union count reconciliation
# - cleaning_audit (DataFrame): reconciliation summary

Month-range filter applied (202503 to 202602): kept 5,601,626 / 5,601,662 rows (removed 36).

Station-name cleanup summary:


,field,rows_changed,rows_null_after_cleanup
0,start_station_name,79942,1192510
1,end_station_name,82698,1254426


Removed configured characters by unique character:


,removed_character,removed_count,unicode_codepoint
0,*,110525,U+002A


Before/after samples by removed character (up to configured sample limit per character):


...

,A/B,start_station_name,start_lat,start_lng,end_station_name,end_lat,end_lng
0,A (dirty),N Green St & W Lake St,41.885579,-87.648484,<NA>,41.910000,-87.630000
1,A (dirty),<NA>,41.930000,-87.650000,<NA>,41.930000,-87.680000
2,A (dirty),Western Ave & Leland Ave,41.966400,-87.688704,<NA>,41.960000,-87.680000
3,B (clean),Woodlawn Ave & 55th St,41.795264,-87.596471,Blackstone Ave & 59th St,41.787877,-87.590461
4,B (clean),Halsted St & Roscoe St,41.943632,-87.649083,Sheffield Ave & Wrightwood Ave,41.928712,-87.653833
5,B (clean),Michigan Ave & Madison St,41.882134,-87.625125,Streeter Dr/Grand Ave,41.892401,-87.612388



=== Dirty Data Overview ===


,dirty_case,count,pct_of_rows,expected_value,why_considered_dirty
5,ANY dirty row (union of all rules),1879365,33.5503,At least one rule violated,Union count.
3,Missing trip location context (start OR end),1874460,33.4628,All start/end location fields are populated (n...,At least one required location field is missing.
1,Zero ride length,16493,0.2944,ride_length_seconds > 0,Zero-duration trip.
6,Rows flagged by >=2 rules (overlap),11617,0.2074,Prefer one issue per row,Overlap count.
0,Invalid trip duration chronology,29,0.0005,ride_length_seconds > 0 and ended_at >= starte...,Negative duration or end before start.
2,Missing rider type,0,0.0000,"member_casual in {'member','casual'}",Rider segment missing.
4,Duplicate ride_id,0,0.0000,ride_id unique per trip,Duplicate trip IDs.



=== Validation Summary ===


,check,value
0,Total rows,5601626
1,Dirty rows (union),1879365
2,Rows with complete location fields,3727166
3,Rows with missing location fields,1874460
4,Union count consistent,True


### Bad Data Clean Up and Drop
This cell applies validation-rule outcomes to split dirty vs. clean records, runs ride-length outlier diagnostics, and prepares the cleaned dataset for downstream mapping and weather merge.

**Section-aligned script flow (matches the Python sections below)**
1. **Section 1 - Build dirty-row union mask**
   - Builds `combined_dirty_mask` by OR-combining all rule masks from `dirty_rules`.

2. **Section 2 - Save flagged rows**
   - Stores flagged records in `dirty_data` for audit and QA review.

3. **Section 3 - Remove flagged rows**
   - Filters `df` to non-flagged rows and synchronizes `clean_data` to this cleaned state.

4. **Section 4 - Print cleanup counts**
   - Prints original shape, cleaned shape, and total rows removed.

5. **Section 5 - Display removed-row samples**
   - Displays a quick sample of removed rows using key trip columns when available.

6. **Section 6 - Outlier diagnostics (`ride_length_seconds`)**
   - Initializes diagnostics fields on `clean_data`: `ride_length_7d_avg_seconds`, `ride_length_mad_zscore`, `ride_length_outlier`.
   - **Section 6.1** computes trailing 7-day rolling averages by rider segment (`member_casual`).
   - **Section 6.2** computes Modified Z-Score outliers by segment with global fallback when segment size is small or MAD is invalid/zero.
   - **Section 6.3** applies optional removal controlled by `remove_flagged_ride_length_outliers` and also builds `non_outlier_clean_data`.
   - Synchronizes `df` with the post-diagnostic `clean_data`.

7. **Section 7 - Drop legacy timestamp column**
   - Drops `ended_at` from `df`, `clean_data`, and `non_outlier_clean_data` because `ride_length_seconds` is already the downstream duration field.

**Outputs from this cell**
- `combined_dirty_mask`: union mask of validation rules.
- `dirty_data`: flagged records retained for QA.
- `clean_data`: cleaned dataset after rule filtering and optional outlier removal.
- `ride_length_7d_avg_seconds`, `ride_length_mad_zscore`, `ride_length_outlier`: outlier diagnostics fields.
- `non_outlier_clean_data`: clean subset where `ride_length_outlier` is False.
- `df`: main flow dataset synchronized to the final cleaned state.


In [36]:
# ================================================================================
# BAD DATA CLEAN UP AND DROP
# ================================================================================
# Purpose: remove rows flagged by validation rules, then run ride-length outlier diagnostics.

# Bad Data Clean Up and Drop
# ================================================================================
# SECTION 1: BUILD DIRTY-ROW UNION MASK
# ================================================================================
# Build a row-level union mask from all dirty rules (index-aligned to current df).
combined_dirty_mask = np.logical_or.reduce([
    rule["mask"].reindex(df.index, fill_value=False).to_numpy()
    for rule in dirty_rules.values()
])

# ================================================================================
# SECTION 2: SAVE FLAGGED ROWS
# ================================================================================
# Save flagged rows to dirty_data for audit/sample review.
dirty_data = df.loc[combined_dirty_mask].copy()

# ================================================================================
# SECTION 3: REMOVE FLAGGED ROWS
# ================================================================================
# Remove flagged rows and keep only clean records in df.
original_shape = df.shape
df = df.loc[~combined_dirty_mask].copy()

# Keep clean_data synchronized with the cleaned df state.
clean_data = df.copy()

# ================================================================================
# SECTION 4: PRINT CLEANUP COUNTS
# ================================================================================
# Print original shape, cleaned shape, and rows removed.
print("Original shape:", original_shape)
print("Cleaned shape: ", df.shape)
print("Rows removed:  ", int(combined_dirty_mask.sum()))

# ================================================================================
# SECTION 5: DISPLAY REMOVED-ROW SAMPLES
# ================================================================================
# Display a sample of removed rows (only columns that currently exist).
preferred_cols_to_show = ['ride_id', 'started_at', 'ended_at', 'ride_length_seconds', 'member_casual']
cols_to_show = [c for c in preferred_cols_to_show if c in dirty_data.columns]
display(dirty_data[cols_to_show].head(10))

# ================================================================================
# SECTION 6: OUTLIER DIAGNOSTICS
# ================================================================================
# Outlier diagnostics for ride_length_seconds.
# - Uses trailing 7-day rolling average as contextual seasonality-aware reference.
# - Uses Modified Z-Score (MAD) by rider segment (member/casual).
mad_threshold = 3.5
mad_min_group_size = 30
# Remove Flag behavior (set in setup cell):
# - True  -> remove outlier rows from clean_data
# - False -> keep all rows and only flag outliers in ride_length_outlier
if 'remove_flagged_ride_length_outliers' not in globals():
    remove_flagged_ride_length_outliers = False

clean_data = clean_data.sort_values('started_at').copy()
clean_data['ride_length_7d_avg_seconds'] = np.nan
clean_data['ride_length_mad_zscore'] = np.nan
clean_data['ride_length_outlier'] = False

# -------------------------------------------------------------------------------
# SECTION 6.1: 7-DAY ROLLING CONTEXT BY RIDER SEGMENT
# -------------------------------------------------------------------------------
for segment, seg_idx in clean_data.groupby('member_casual', dropna=False).groups.items():
    seg_df = clean_data.loc[seg_idx].sort_values('started_at')
    seg_roll = (
        seg_df
        .set_index('started_at')['ride_length_seconds']
        .rolling('7D', min_periods=1)
        .mean()
    )
    clean_data.loc[seg_df.index, 'ride_length_7d_avg_seconds'] = seg_roll.to_numpy()

# -------------------------------------------------------------------------------
# SECTION 6.2: SEGMENT-LEVEL MAD MODIFIED Z-SCORE
# -------------------------------------------------------------------------------
global_vals = clean_data['ride_length_seconds'].dropna().astype(float)
global_median = global_vals.median()
global_mad = np.median(np.abs(global_vals - global_median)) if len(global_vals) else np.nan

for segment, seg_idx in clean_data.groupby('member_casual', dropna=False).groups.items():
    seg_vals = clean_data.loc[seg_idx, 'ride_length_seconds'].astype(float)
    seg_non_null = seg_vals.dropna()

    use_global = len(seg_non_null) < mad_min_group_size
    if not use_global:
        seg_median = seg_non_null.median()
        seg_mad = np.median(np.abs(seg_non_null - seg_median))
        if pd.isna(seg_mad) or seg_mad == 0:
            use_global = True

    if use_global:
        seg_median = global_median
        seg_mad = global_mad

    # If MAD is still zero/NaN, use deterministic fallback for constant distributions.
    if pd.isna(seg_mad) or seg_mad == 0:
        zscores = pd.Series(np.nan, index=seg_vals.index, dtype='float64')
        valid = seg_vals.notna()
        zscores.loc[valid] = np.where(seg_vals.loc[valid] == seg_median, 0.0, np.inf)
    else:
        zscores = 0.6745 * (seg_vals - seg_median) / seg_mad

    clean_data.loc[seg_idx, 'ride_length_mad_zscore'] = zscores
    clean_data.loc[seg_idx, 'ride_length_outlier'] = (
        zscores.abs() > mad_threshold
    ).fillna(False)

# -------------------------------------------------------------------------------
# SECTION 6.3: OPTIONAL OUTLIER REMOVAL
# -------------------------------------------------------------------------------
flagged_outlier_count = int(clean_data['ride_length_outlier'].sum())
if remove_flagged_ride_length_outliers:
    clean_data = clean_data.loc[~clean_data['ride_length_outlier']].copy()
    print(f"Remove Flag = True -> removed ride-length outliers: {flagged_outlier_count:,}")
else:
    print(f"Remove Flag = False -> outliers flagged in 'ride_length_outlier': {flagged_outlier_count:,}")

# Optional non-outlier view for downstream comparisons.
non_outlier_clean_data = clean_data.loc[~clean_data['ride_length_outlier']].copy()

print(f"Outlier threshold (Modified Z-Score): {mad_threshold}")
print(f"Rows retained in main clean_data: {len(clean_data):,}")
print(f"Rows in non_outlier_clean_data: {len(non_outlier_clean_data):,}")

# Keep df synchronized with clean_data so downstream cells include outlier diagnostics.
df = clean_data.copy()

# ================================================================================
# SECTION 7: DROP LEGACY TIMESTAMP COLUMN
# ================================================================================
# Drop ended_at — ride_length_seconds is already derived and ended_at is no longer required downstream.
if 'ended_at' in df.columns:
    df = df.drop(columns=['ended_at'])
if 'ended_at' in clean_data.columns:
    clean_data = clean_data.drop(columns=['ended_at'])
if 'ended_at' in non_outlier_clean_data.columns:
    non_outlier_clean_data = non_outlier_clean_data.drop(columns=['ended_at'])

print("\nColumns after drop:", df.columns.tolist())


Original shape: (5601626, 16)
Cleaned shape:  (3722261, 16)
Rows removed:   1879365


,ride_id,started_at,ended_at,ride_length_seconds,member_casual
37,8A92C72E248830F5,2025-03-01 00:00:00,2025-03-01 00:15:00,850,member
42,444E734A2034F5EA,2025-03-01 00:05:00,2025-03-01 00:10:00,500,casual
43,7B5A5D2AC970E669,2025-03-01 00:05:00,2025-03-01 00:05:00,210,member
46,1F105646D53608E1,2025-03-01 00:05:00,2025-03-01 00:10:00,420,casual
47,00E3B7301334673A,2025-03-01 00:05:00,2025-03-01 00:10:00,230,member
55,99EB37398DCF822E,2025-03-01 00:05:00,2025-03-01 00:10:00,150,member
56,FC23F04CCA9CFB3B,2025-03-01 00:05:00,2025-03-01 00:10:00,270,member
59,3C7C3D5B75F548D6,2025-03-01 00:05:00,2025-03-01 00:15:00,450,member
60,EE435E4847901B64,2025-03-01 00:05:00,2025-03-01 00:10:00,360,casual
67,EAEB1E30DC1C9261,2025-03-01 00:10:00,2025-03-01 00:30:00,1050,casual


Remove Flag = False -> outliers flagged in 'ride_length_outlier': 258,859
Outlier threshold (Modified Z-Score): 3.5
Rows retained in main clean_data: 3,722,261
Rows in non_outlier_clean_data: 3,463,402

Columns after drop: ['ride_id', 'rideable_type', 'started_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'ride_length_seconds', 'dt_start_hour', 'day_of_week', 'ride_length_7d_avg_seconds', 'ride_length_mad_zscore', 'ride_length_outlier']


### Coordinate Vector Mapping
This cell converts raw trip start/end coordinates into stable grid-based vector keys by building a bounded map, defining a grid lock, and mapping each point to a grid-cell center.

**Section-aligned script flow (matches the Python sections below)**
1. **Section 1 - `create_vector_map(df)`**
   - Computes the geographic bounding box from `start_lat`, `start_lng`, `end_lat`, and `end_lng`.
   - Applies a small outer buffer and returns bounded map limits (`lat_north`, `lat_south`, `lng_west`, `lng_east`).

2. **Section 2 - `create_grid_lock(vector_map, grid_resolution)`**
   - Builds grid metadata from the bounded map and selected `grid_resolution`.
   - Computes grid step counts (`lat_steps`, `lng_steps`), a resolution scalar, and approximate cell edge/area metrics.
   - Returns `grid_lock`, which is used as the mapping control object.

3. **Section 3 - `apply_grid_mapping(df, grid_lock)`**
   - Maps start/end coordinates to nearest grid-cell centers using floor-based binning from the grid origin.
   - Creates mapped columns: `start-lat_vmap`, `start-lng_vmap`, `end-lat_vmap`, `end-lng_vmap`.
   - Drops temporary center columns and returns the mapped DataFrame.

4. **Execution sequence in this cell**
   - Creates `vector_map` -> sets `grid_resolution` -> creates `grid_lock` -> applies mapping -> replaces `df` with mapped output.

**Outputs from this cell**
- `vector_map`: buffered coordinate bounds for the study area.
- `grid_lock`: grid configuration and diagnostic metadata.
- `df`: mapped dataset with stable vector key columns for start/end coordinates.
- `start-lat_vmap`, `start-lng_vmap`, `end-lat_vmap`, `end-lng_vmap`: downstream-ready spatial grouping keys.

In [37]:
# ================================================================================
# COORDINATE VECTOR MAPPING
# ================================================================================
# Purpose: convert raw lat/lng values into stable grid-based coordinate keys.

# Coordinate mapping

# ================================================================================
# SECTION 1: CREATE VECTOR MAP (BOUNDING BOX)
# ================================================================================
def create_vector_map(df):
    """Step 1: Compute bounding box from start/end lat/lng values."""
    lat_min = min(df['start_lat'].min(), df['end_lat'].min())
    lat_max = max(df['start_lat'].max(), df['end_lat'].max())
    lng_min = min(df['start_lng'].min(), df['end_lng'].min())
    lng_max = max(df['start_lng'].max(), df['end_lng'].max())

    buffer = 0.02
    vector_map = {
        'lat_north': round(lat_max + buffer, 4),
        'lat_south': round(lat_min - buffer, 4),
        'lng_west': round(lng_min - buffer, 4),
        'lng_east': round(lng_max + buffer, 4),
    }
    print('Vector map (bounding box):', vector_map)
    return vector_map


# ================================================================================
# SECTION 2: CREATE GRID LOCK
# ================================================================================
def create_grid_lock(vector_map, grid_resolution=0.01):
    """Step 2: Build grid lock using resolution in degrees.

    Tuning guide:
    - Lower grid_resolution -> finer grid (more detail, more cells).
    - Higher grid_resolution -> coarser grid (less detail, fewer cells).
    """
    lat_steps = int((vector_map['lat_north'] - vector_map['lat_south']) / grid_resolution) + 1
    lng_steps = int((vector_map['lng_east'] - vector_map['lng_west']) / grid_resolution) + 1

    # Scalar view of the chosen grid resolution.
    # Example: 0.01 -> scalar 100, 0.005 -> scalar 200.
    grid_scalar = 1.0 / grid_resolution

    # Approximate cell edge length assuming 1 degree latitude ~= 111 km ~= 68.973 miles.
    approx_edge_km = 111.0 * grid_resolution
    approx_edge_m = approx_edge_km * 1000.0
    approx_edge_miles = 68.973 * grid_resolution
    approx_edge_feet = approx_edge_miles * 5280.0

    # Approximate cell area as a square footprint from the edge-length estimate.
    approx_area_sq_km = approx_edge_km ** 2
    approx_area_sq_m = approx_edge_m ** 2
    approx_area_sq_miles = approx_edge_miles ** 2
    approx_area_sq_feet = approx_edge_feet ** 2

    grid_lock = {
        'grid_resolution': grid_resolution,
        'grid_scalar': grid_scalar,
        'approx_edge_km': approx_edge_km,
        'approx_edge_m': approx_edge_m,
        'approx_edge_miles': approx_edge_miles,
        'approx_edge_feet': approx_edge_feet,
        'approx_area_sq_km': approx_area_sq_km,
        'approx_area_sq_m': approx_area_sq_m,
        'approx_area_sq_miles': approx_area_sq_miles,
        'approx_area_sq_feet': approx_area_sq_feet,
        'lat_north': vector_map['lat_north'],
        'lat_south': vector_map['lat_south'],
        'lng_west': vector_map['lng_west'],
        'lng_east': vector_map['lng_east'],
        'lat_steps': lat_steps,
        'lng_steps': lng_steps,
    }

    print(f"Grid lock created: {lat_steps} lat steps x {lng_steps} lng steps at {grid_resolution} deg resolution")
    print(f"Resolution scalar (1 / grid_resolution): {grid_scalar:.2f}")
    print(
        "Cell edge length estimate (1D side length): "
        f"{approx_edge_km:.6f} km | {approx_edge_m:.2f} m | "
        f"{approx_edge_miles:.6f} miles | {approx_edge_feet:.2f} ft"
    )
    print(
        "Cell area estimate (2D surface footprint): "
        f"{approx_area_sq_km:.8f} sq km | {approx_area_sq_m:.2f} sq m | "
        f"{approx_area_sq_miles:.8f} sq miles | {approx_area_sq_feet:.2f} sq ft"
    )
    return grid_lock


# ================================================================================
# SECTION 3: APPLY GRID MAPPING
# ================================================================================
def apply_grid_mapping(df, grid_lock):
    """Step 3: Map coordinates to cell centers and create separate vmap columns per axis."""
    mapped = df.copy()
    res = grid_lock['grid_resolution']

    lat_origin = grid_lock['lat_south']
    lng_origin = grid_lock['lng_west']

    # Bin index from grid origin, then shift by +0.5 to center of each cell.
    start_lat_idx = np.floor((mapped['start_lat'] - lat_origin) / res)
    start_lng_idx = np.floor((mapped['start_lng'] - lng_origin) / res)
    end_lat_idx = np.floor((mapped['end_lat'] - lat_origin) / res)
    end_lng_idx = np.floor((mapped['end_lng'] - lng_origin) / res)

    mapped['start_lat_center'] = lat_origin + (start_lat_idx + 0.5) * res
    mapped['start_lng_center'] = lng_origin + (start_lng_idx + 0.5) * res
    mapped['end_lat_center'] = lat_origin + (end_lat_idx + 0.5) * res
    mapped['end_lng_center'] = lng_origin + (end_lng_idx + 0.5) * res

    # Separate vmap columns keep vector-mapped coordinates distinct from raw lat/lng fields.
    mapped['start-lat_vmap'] = mapped['start_lat_center'].round(6)
    mapped['start-lng_vmap'] = mapped['start_lng_center'].round(6)
    mapped['end-lat_vmap'] = mapped['end_lat_center'].round(6)
    mapped['end-lng_vmap'] = mapped['end_lng_center'].round(6)

    mapped = mapped.drop(columns=['start_lat_center', 'start_lng_center', 'end_lat_center', 'end_lng_center'])
    print(f"Mapping complete: {mapped['start-lat_vmap'].nunique():,} unique start grid-cell centers")
    return mapped


# Apply coordinate mapping to cleaned df and keep df as the main flow variable
vector_map = create_vector_map(df)

# Change this value to control map granularity.
# Lower = finer detail, Higher = coarser detail.
grid_resolution = 0.00005
grid_lock = create_grid_lock(vector_map, grid_resolution=grid_resolution)

df = apply_grid_mapping(df, grid_lock)
df[['start-lat_vmap', 'start-lng_vmap', 'end-lat_vmap', 'end-lng_vmap']].head()

Vector map (bounding box): {'lat_north': 42.0849, 'lat_south': 41.6285, 'lng_west': -87.8641, 'lng_east': -87.5082}
Grid lock created: 9128 lat steps x 7118 lng steps at 5e-05 deg resolution
Resolution scalar (1 / grid_resolution): 20000.00
Cell edge length estimate (1D side length): 0.005550 km | 5.55 m | 0.003449 miles | 18.21 ft
Cell area estimate (2D surface footprint): 0.00003080 sq km | 30.80 sq m | 0.00001189 sq miles | 331.56 sq ft
Mapping complete: 3,108 unique start grid-cell centers


,start-lat_vmap,start-lng_vmap,end-lat_vmap,end-lng_vmap
36,41.795275,-87.596475,41.787875,-87.590475
38,41.943625,-87.649075,41.928725,-87.653825
62,41.896725,-87.630875,41.903225,-87.634325
61,41.939725,-87.658875,41.918325,-87.636275
58,41.940225,-87.652925,41.929525,-87.643125


In [38]:
# ================================================================================
# COORDINATE MAPPING CHECKPOINT
# ================================================================================
# Coordinate-mapping checkpoint: drop raw lat/lng columns, reorder columns, and confirm vmap columns.
# Outputs:
# - df: working dataset with raw coordinate columns removed and columns reordered for downstream use.
# - vmap_cols: list of the four new grid-mapped coordinate columns.


# Validate expected vmap columns before attempting reordering.
vmap_cols = ['start-lat_vmap', 'start-lng_vmap', 'end-lat_vmap', 'end-lng_vmap']
missing_vmap = [c for c in vmap_cols if c not in df.columns]
if missing_vmap:
    raise ValueError(
        "Coordinate mapping columns are missing. Run the previous 'Coordinate mapping' cell first. "
        f"Missing columns: {missing_vmap}"
    )

# ================================================================================
# SECTION 1: DROP RAW COORDINATE COLUMNS
# ================================================================================
# Drop raw coordinate columns now superseded by vmap columns.
cols_to_drop_coords = ['start_lat', 'start_lng', 'end_lat', 'end_lng']
df = df.drop(columns=[c for c in cols_to_drop_coords if c in df.columns])

# ================================================================================
# SECTION 2: REORDER COLUMNS
# ================================================================================
# Reorder columns; pass-through any unexpected columns at the end.
ordered_cols = [
    'started_at',
    'day_of_week',
    'start_station_name',
    'start_station_id',
    'start-lat_vmap',
    'start-lng_vmap',
    'end_station_name',
    'end_station_id',
    'end-lat_vmap',
    'end-lng_vmap',
    'ride_id',
    'member_casual',
    'rideable_type',
    'ride_length_seconds',
    'ride_length_7d_avg_seconds',
    'ride_length_mad_zscore',
    'ride_length_outlier',
    'dt_start_hour',
]
remaining_cols = [c for c in df.columns if c not in ordered_cols]
df = df[ordered_cols + remaining_cols]

print(f"df shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

print("\nvmap column preview:")
display(df[vmap_cols].head(10))
display(df[7:17].head())


df shape: 3,722,261 rows x 18 columns

Columns (18):
  started_at
  day_of_week
  start_station_name
  start_station_id
  start-lat_vmap
  start-lng_vmap
  end_station_name
  end_station_id
  end-lat_vmap
  end-lng_vmap
  ride_id
  member_casual
  rideable_type
  ride_length_seconds
  ride_length_7d_avg_seconds
  ride_length_mad_zscore
  ride_length_outlier
  dt_start_hour

vmap column preview:


,start-lat_vmap,start-lng_vmap,end-lat_vmap,end-lng_vmap
36,41.795275,-87.596475,41.787875,-87.590475
38,41.943625,-87.649075,41.928725,-87.653825
62,41.896725,-87.630875,41.903225,-87.634325
61,41.939725,-87.658875,41.918325,-87.636275
58,41.940225,-87.652925,41.929525,-87.643125
54,41.802425,-87.586925,41.795225,-87.580725
53,41.892375,-87.676875,41.898425,-87.686575
52,41.898975,-87.629925,41.882425,-87.639775
51,42.009025,-87.674125,42.007975,-87.665525
57,41.885625,-87.641825,41.885775,-87.651025


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,ride_id,member_casual,rideable_type,ride_length_seconds,ride_length_7d_avg_seconds,ride_length_mad_zscore,ride_length_outlier,dt_start_hour
52,2025-03-01 00:05:00,5,Dearborn Pkwy & Delaware Pl,TA1307000128,41.898975,-87.629925,Canal St & Madison St,13341,41.882425,-87.639775,8A27F4E7D27C870A,member,classic_bike,890,380.000000,0.998260,False,2025-03-01
51,2025-03-01 00:05:00,5,Clark St & Lunt Ave,KA1504000162,42.009025,-87.674125,Glenwood Ave & Morse Ave,KA1504000175,42.007975,-87.665525,ADDB2D3718363D1A,member,electric_bike,160,343.333333,-0.971280,False,2025-03-01
57,2025-03-01 00:05:00,5,Clinton St & Lake St,13021,41.885625,-87.641825,Sangamon St & Lake St,TA1306000015,41.885775,-87.651025,A9B7058ACF1B24A3,casual,classic_bike,320,357.500000,-0.740305,False,2025-03-01
49,2025-03-01 00:05:00,5,Clark St & Montrose Ave,KA1503000022,41.961575,-87.666025,Clarendon Ave & Gordon Ter,13379,41.957875,-87.649525,253E9131AFA9F27B,member,electric_bike,270,332.857143,-0.674500,False,2025-03-01
48,2025-03-01 00:05:00,5,Clark St & Lunt Ave,KA1504000162,42.009025,-87.674125,Glenwood Ave & Morse Ave,KA1504000175,42.007975,-87.665525,83C885DFCF77D7AD,member,electric_bike,170,312.500000,-0.944300,False,2025-03-01


### Project Merge with Weather Data 
This cell maps trip-origin grid points to their nearest weather station, merges hourly weather onto trips, audits unmatched rows, and produces the final weather-enriched dataset.

**Section-aligned script flow (matches the Python sections below)**
1. **Section 1 - Parse datetime join keys**
   - Parses `weather_df['time']` and `df['dt_start_hour']` to datetime for safe key-based joining.

2. **Section 2 - Drop previous weather enrichment columns**
   - Removes prior weather merge columns so the cell can be rerun without duplicate fields.

3. **Section 3 - Build imperial weather fields**
   - Creates converted weather fields (`temperature_7ft_f`, `precipitation_in`, `rain_in`, `snowfall_in`, `wind_speed_33ft_mph`) and aligned humidity/wind/cloud columns.

4. **Section 4 - Extract unique start grid points**
   - Builds unique trip-origin points from `start-lat_vmap` and `start-lng_vmap`.

5. **Section 5 - Build start-point x weather-location candidates**
   - Creates a Cartesian candidate table between trip start points and weather stations.
   - **Section 5.1** converts coordinates to radians.
   - **Section 5.2** computes haversine distance in km and miles.

6. **Section 6 - Select nearest weather location per start point**
   - Keeps one nearest station per unique start grid point and materializes `start_weather_map`.
|
7. **Section 7 - Attach weather location metadata to trips**
   - Left-merges `start_weather_map` into trip rows, preserving original trip count.

8. **Section 8 - Merge hourly weather**
   - Left-merges hourly weather using `weather_location_id` + `dt_start_hour`.

9. **Section 9 - Classify unmatched weather rows**
   - Isolates rows with missing weather and assigns `unmatched_reason` using ordered rules.
   - Builds summary and reason-guide diagnostics tables.

10. **Section 10 - Build geo-mapping QA tables**
   - Builds station assignment and distance QA outputs (`weather_location_assignment_summary`, `weather_geo_mapping_audit`).

11. **Section 11 - Display merge audit and diagnostics**
   - Compiles `weather_merge_audit` and displays matched/unmatched and mapping QA checks.

12. **Section 12 - Build final weather-enriched output**
   - Creates `df_weather_merge` and drops any legacy duplicate weather columns.

**Important handling behavior**
- All merges are left joins on trips, so trip row count is preserved.
- Rows without matched weather are retained and documented via `unmatched_reason`.

**Outputs from this cell**
- `start_weather_map`: nearest weather station mapping per unique trip-origin grid point.
- `unmatched_weather_rows_df`, `unmatched_weather_summary`, `unmatched_weather_summary_detailed`: unmatched diagnostics.
- `weather_location_assignment_summary`, `weather_geo_mapping_audit`: mapping QA outputs.
- `weather_merge_audit`: row-count and coverage validation summary.
- `df_weather_merge`: final weather-enriched trip dataset.

In [39]:
# ================================================================================
# PROJECT MERGE WITH WEATHER DATA
# ================================================================================
# Purpose: map trip start points to nearest weather station and merge hourly weather by trip hour.

# ## Weather Data Processing and Merge
# ================================================================================
# SECTION 1: PARSE DATETIME JOIN KEYS
# ================================================================================
# Parse weather timestamps and trip start-hour values into join-ready datetime fields.
weather_df = weather_df.copy()
weather_locations_df = weather_locations_df.copy()

weather_df['time'] = pd.to_datetime(weather_df['time'], errors='coerce')
df['dt_start_hour'] = pd.to_datetime(df['dt_start_hour'], errors='coerce')

# ================================================================================
# SECTION 2: DROP PREVIOUS WEATHER ENRICHMENT COLUMNS
# ================================================================================
# Drop previously merged weather-enrichment columns so this cell can be rerun safely.
weather_enrichment_cols = [
    'weather_location_id',
    'weather_latitude',
    'weather_longitude',
    'distance_km',
    'distance_miles',
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'precipitation (mm)',
    'rain (mm)',
    'snowfall (cm)',
    'wind_speed_10m (km/h)',
    'wind_direction_10m (°)',
    'cloud_cover (%)',
    'temperature_7ft_f',
    'precipitation_in',
    'rain_in',
    'snowfall_in',
    'wind_speed_33ft_mph',
    'relative_humidity_7ft_pct',
    'wind_direction_33ft_deg',
    'cloud_cover_pct',
]
existing_weather_cols = [col for col in weather_enrichment_cols if col in df.columns]
if existing_weather_cols:
    df = df.drop(columns=existing_weather_cols)

# ================================================================================
# SECTION 3: BUILD IMPERIAL WEATHER FIELDS
# ================================================================================
# Build imperial weather fields and feet-based height labels used in downstream outputs.
# Height label conversions used in column names:
# - 2 meters ~= 6.56 feet -> labeled as 7ft
# - 10 meters ~= 32.81 feet -> labeled as 33ft
weather_df['temperature_7ft_f'] = (weather_df['temperature_2m (°C)'] * 9 / 5) + 32
weather_df['precipitation_in'] = weather_df['precipitation (mm)'] / 25.4
weather_df['rain_in'] = weather_df['rain (mm)'] / 25.4
weather_df['snowfall_in'] = weather_df['snowfall (cm)'] / 2.54
weather_df['wind_speed_33ft_mph'] = weather_df['wind_speed_10m (km/h)'] * 0.621371
weather_df['relative_humidity_7ft_pct'] = weather_df['relative_humidity_2m (%)']
weather_df['wind_direction_33ft_deg'] = weather_df['wind_direction_10m (°)']
weather_df['cloud_cover_pct'] = weather_df['cloud_cover (%)']

# ================================================================================
# SECTION 4: EXTRACT UNIQUE START GRID POINTS
# ================================================================================
# Extract unique trip start grid points from separate vmap columns.
start_points = (
    df[['start-lat_vmap', 'start-lng_vmap']]
    .drop_duplicates()
    .rename(columns={
        'start-lat_vmap': 'start_lat_vmap',
        'start-lng_vmap': 'start_lng_vmap',
    })
    .copy()
)

# ================================================================================
# SECTION 5: BUILD START-POINT X WEATHER-LOCATION CANDIDATES
# ================================================================================
# Build all start-point x weather-location candidates for nearest-station selection.
weather_locations = weather_locations_df[['location_id', 'latitude', 'longitude']].copy()
start_points['__merge_key'] = 1
weather_locations['__merge_key'] = 1
start_weather_candidates = start_points.merge(weather_locations, on='__merge_key', how='left')

# -------------------------------------------------------------------------------
# SECTION 5.1: CONVERT COORDINATES TO RADIANS
# -------------------------------------------------------------------------------
# Convert degrees to radians so haversine distance can be calculated correctly.
earth_radius_km = 6371.0
start_lat_rad = np.radians(start_weather_candidates['start_lat_vmap'])
start_lng_rad = np.radians(start_weather_candidates['start_lng_vmap'])
weather_lat_rad = np.radians(start_weather_candidates['latitude'])
weather_lng_rad = np.radians(start_weather_candidates['longitude'])

# -------------------------------------------------------------------------------
# SECTION 5.2: COMPUTE HAVERSINE DISTANCES
# -------------------------------------------------------------------------------
# Compute haversine distance between each trip start point and candidate weather location.
delta_lat = weather_lat_rad - start_lat_rad
delta_lng = weather_lng_rad - start_lng_rad
haversine_a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(start_lat_rad) * np.cos(weather_lat_rad) * np.sin(delta_lng / 2) ** 2
)
haversine_c = 2 * np.arctan2(np.sqrt(haversine_a), np.sqrt(1 - haversine_a))
start_weather_candidates['distance_km'] = earth_radius_km * haversine_c
start_weather_candidates['distance_miles'] = start_weather_candidates['distance_km'] * 0.621371

# ================================================================================
# SECTION 6: SELECT NEAREST WEATHER LOCATION PER START POINT
# ================================================================================
# Keep the single nearest weather location for each unique trip start grid point.
nearest_idx = (
    start_weather_candidates
    .groupby(['start_lat_vmap', 'start_lng_vmap'])['distance_km']
    .idxmin()
)
start_weather_map = (
    start_weather_candidates.loc[nearest_idx, [
        'start_lat_vmap',
        'start_lng_vmap',
        'location_id',
        'latitude',
        'longitude',
        'distance_km',
        'distance_miles',
    ]]
    .rename(columns={
        'start_lat_vmap': 'start-lat_vmap',
        'start_lng_vmap': 'start-lng_vmap',
        'location_id': 'weather_location_id',
        'latitude': 'weather_latitude',
        'longitude': 'weather_longitude',
    })
    .reset_index(drop=True)
)

# ================================================================================
# SECTION 7: ATTACH WEATHER LOCATION METADATA TO TRIPS
# ================================================================================
# Attach nearest weather location metadata and proximity fields to each bike-trip row.
original_row_count = len(df)
df = df.merge(start_weather_map, on=['start-lat_vmap', 'start-lng_vmap'], how='left')

# ================================================================================
# SECTION 8: MERGE HOURLY WEATHER
# ================================================================================
# Merge hourly weather by weather_location_id + dt_start_hour using imperial/dimensionless outputs.
# Left join preserves all bike rows even when hourly weather is unavailable.
weather_merge_cols = [
    'location_id',
    'time',
    'temperature_7ft_f',
    'relative_humidity_7ft_pct',
    'precipitation_in',
    'rain_in',
    'snowfall_in',
    'wind_speed_33ft_mph',
    'wind_direction_33ft_deg',
    'cloud_cover_pct',
]

df = df.merge(
    weather_df[weather_merge_cols],
    left_on=['weather_location_id', 'dt_start_hour'],
    right_on=['location_id', 'time'],
    how='left',
).drop(columns=['location_id', 'time'])

# ================================================================================
# SECTION 9: CLASSIFY UNMATCHED WEATHER ROWS
# ================================================================================
# Identify unmatched rows and classify why weather data did not join.
weather_coverage_start = weather_df['time'].min()
weather_coverage_end = weather_df['time'].max()

# Unmatched rows are intentionally retained in df; this QA frame isolates them for diagnostics only.
unmatched_weather_rows_df = df.loc[df['temperature_7ft_f'].isna()].copy()
# Classification is ordered by priority (first true condition wins).
unmatched_weather_rows_df['unmatched_reason'] = np.select(
    [
        unmatched_weather_rows_df['dt_start_hour'].isna(),
        unmatched_weather_rows_df['weather_location_id'].isna(),
        unmatched_weather_rows_df['dt_start_hour'] < weather_coverage_start,
        unmatched_weather_rows_df['dt_start_hour'] > weather_coverage_end,
    ],
    [
        'Missing dt_start_hour',
        'Missing weather location mapping',
        'Before weather coverage window',
        'After weather coverage window',
    ],
    default='No hourly weather row for mapped location and hour',
)

unmatched_weather_summary = (
    unmatched_weather_rows_df['unmatched_reason']
    .value_counts(dropna=False)
    .rename_axis('unmatched_reason')
    .reset_index(name='row_count')
)

# Add condition-level definitions and recommended next actions for each unmatched reason.
unmatched_reason_guide = pd.DataFrame([
    {
        'unmatched_reason': 'Missing dt_start_hour',
        'condition_detail': 'Trip hour key is null after datetime parsing/rounding.',
        'suggested_fix': 'Review started_at parsing and dt_start_hour derivation; repair or drop rows with invalid timestamps.',
    },
    {
        'unmatched_reason': 'Missing weather location mapping',
        'condition_detail': 'No nearest weather location_id was attached to the start grid point.',
        'suggested_fix': 'Validate start-lat_vmap/start-lng_vmap values and weather location table coverage for the study area.',
    },
    {
        'unmatched_reason': 'Before weather coverage window',
        'condition_detail': 'Trip hour is earlier than the minimum weather_df time.',
        'suggested_fix': 'Expand weather_start_date backward or limit bike rows to covered dates.',
    },
    {
        'unmatched_reason': 'After weather coverage window',
        'condition_detail': 'Trip hour is later than the maximum weather_df time.',
        'suggested_fix': 'Expand weather_end_date forward or limit bike rows to covered dates.',
    },
    {
        'unmatched_reason': 'No hourly weather row for mapped location and hour',
        'condition_detail': 'Station is mapped but no hourly observation exists for that exact hour.',
        'suggested_fix': 'Inspect weather source gaps, timezone alignment, and consider fallback joins (nearest available hour).',
    },
])

unmatched_weather_summary_detailed = (
    unmatched_weather_summary
    .merge(unmatched_reason_guide, on='unmatched_reason', how='left')
)

# ================================================================================
# SECTION 10: BUILD GEO-MAPPING QA TABLES
# ================================================================================
# Build geo-mapping QA tables for station assignment counts and point-to-station proximity.
weather_location_assignment_summary = (
    df.groupby('weather_location_id', dropna=False)
    .agg(
        mapped_rows=('weather_location_id', 'size'),
        avg_distance_km=('distance_km', 'mean'),
        max_distance_km=('distance_km', 'max'),
    )
    .reset_index()
)

unique_start_points_by_location = (
    df.groupby('weather_location_id', dropna=False)[['start-lat_vmap', 'start-lng_vmap']]
    .apply(lambda g: g.drop_duplicates().shape[0])
    .rename('unique_start_points')
    .reset_index()
)

weather_location_assignment_summary = (
    weather_location_assignment_summary
    .merge(unique_start_points_by_location, on='weather_location_id', how='left')
    .sort_values('mapped_rows', ascending=False)
)

weather_geo_mapping_audit = (
    start_weather_map[[
        'start-lat_vmap',
        'start-lng_vmap',
        'weather_location_id',
        'weather_latitude',
        'weather_longitude',
        'distance_km',
        'distance_miles',
    ]]
    .sort_values('distance_km', ascending=False)
    .reset_index(drop=True)
)

# ================================================================================
# SECTION 11: DISPLAY MERGE AUDIT & DIAGNOSTICS
# ================================================================================
# Compile and display merge audit + unmatched diagnostics + mapping QA tables.
matched_weather_rows = int(df['temperature_7ft_f'].notna().sum())
missing_weather_rows = int(df['temperature_7ft_f'].isna().sum())

weather_merge_audit = pd.DataFrame([
    {'check': 'Original bike rows', 'value': original_row_count},
    {'check': 'Rows after weather merge', 'value': len(df)},
    {'check': 'Row count preserved', 'value': len(df) == original_row_count},
    {'check': 'Rows with matched weather', 'value': matched_weather_rows},
    {'check': 'Rows missing weather', 'value': missing_weather_rows},
    {'check': 'Unique mapped weather locations', 'value': int(df['weather_location_id'].nunique())},
    {'check': 'Weather coverage start', 'value': weather_coverage_start},
    {'check': 'Weather coverage end', 'value': weather_coverage_end},
])

print("=== Weather Merge Audit ===")
print(
    f"Review of merge integrity and coverage.\n"
    f"Configured weather_start_date: {weather_start_date}\n"
    f"Observed weather coverage window: {weather_coverage_start} to {weather_coverage_end}."
)
display(weather_merge_audit)

print("=== Unmatched Weather Diagnostics ===")
if unmatched_weather_rows_df.empty:
    print("All bike rows successfully matched to hourly weather data.\n")
else:
    print(
        "⚠️ Some bike rows did not match weather data. "
        "See reason counts, definitions, and suggested fixes below."
    )
    display(unmatched_weather_summary_detailed)
    print(
        "Sample unmatched rows (first 20): includes trip time key, mapped weather location,\n"
        "distance to station, and the classified unmatched_reason."
    )
    display(unmatched_weather_rows_df[[
        'started_at',
        'dt_start_hour',
        'start_station_name',
        'start-lat_vmap',
        'start-lng_vmap',
        'weather_location_id',
        'distance_km',
        'unmatched_reason',
    ]].head(20))

print("=== Weather Location Assignment Summary ===")
print(
    "Counts and distance stats by mapped weather location_id.\n"
    "Useful for identifying overused stations or far mappings."
)
display(weather_location_assignment_summary)

print("=== Geo Mapping Audit (Top 20 Farthest Start-Point Mappings) ===")
print(
    "Point-to-station mapping QA: each row is a unique trip start grid point mapped\n"
    "to its nearest weather location, sorted by largest distance_km."
)
display(weather_geo_mapping_audit.head(20))

# ================================================================================
# SECTION 12: BUILD FINAL WEATHER-ENRICHED OUTPUT
# ================================================================================
# Create final weather-enriched output and remove any legacy duplicate weather columns if present.
df_weather_merge = df.copy()

# Drop legacy/duplicate weather columns if present.
cols_to_drop = [
    'distance_km',
    'temperature_2m_f',
    'relative_humidity_2m_pct',
    'wind_speed_10m_mph',
    'wind_direction_10m_deg',
]
df_weather_merge = df_weather_merge.drop(columns=cols_to_drop, errors='ignore')

print("=== Final Weather-Enriched Dataset ===")
print(f"Rows: {len(df_weather_merge):,} | Columns: {df_weather_merge.shape[1]:,}")

# Preview rows with all columns visible.
with pd.option_context(
    'display.max_columns', None,
    'display.width', None,
    'display.max_colwidth', None
):
    display(df_weather_merge.head(10))


=== Weather Merge Audit ===
Review of merge integrity and coverage.
Configured weather_start_date: 2025-02-28
Observed weather coverage window: 2025-02-28 00:00:00 to 2026-02-28 23:00:00.


,check,value
0,Original bike rows,3722261
1,Rows after weather merge,3722261
2,Row count preserved,True
3,Rows with matched weather,3722260
4,Rows missing weather,1
5,Unique mapped weather locations,2
6,Weather coverage start,2025-02-28 00:00:00
7,Weather coverage end,2026-02-28 23:00:00


=== Unmatched Weather Diagnostics ===
⚠️ Some bike rows did not match weather data. See reason counts, definitions, and suggested fixes below.


,unmatched_reason,row_count,condition_detail,suggested_fix
0,After weather coverage window,1,Trip hour is later than the maximum weather_df...,Expand weather_end_date forward or limit bike ...


Sample unmatched rows (first 20): includes trip time key, mapped weather location,
distance to station, and the classified unmatched_reason.


...

,weather_location_id,mapped_rows,avg_distance_km,max_distance_km,unique_start_points
1,1,1954187,27.942492,37.880713,9051
0,0,1768074,27.553709,37.141964,10567


=== Geo Mapping Audit (Top 20 Farthest Start-Point Mappings) ===
Point-to-station mapping QA: each row is a unique trip start grid point mapped
to its nearest weather location, sorted by largest distance_km.


,start-lat_vmap,start-lng_vmap,weather_location_id,weather_latitude,weather_longitude,distance_km,distance_miles
0,41.993025,-87.821675,1,42.07381,-87.37610,37.880713,23.537977
1,41.996975,-87.821325,1,42.07381,-87.37610,37.749699,23.456568
2,41.996925,-87.821025,1,42.07381,-87.37610,37.726839,23.442364
3,41.990625,-87.816925,1,42.07381,-87.37610,37.565201,23.341927
4,42.002325,-87.817075,1,42.07381,-87.37610,37.275028,23.161621
5,42.002375,-87.817075,1,42.07381,-87.37610,37.273829,23.160876
6,41.988575,-87.812525,1,42.07381,-87.37610,37.270969,23.159099
7,42.011475,-87.819025,1,42.07381,-87.37610,37.227019,23.131790
8,42.011475,-87.818225,1,42.07381,-87.37610,37.162113,23.091460
9,42.011525,-87.818225,1,42.07381,-87.37610,37.161063,23.090807


=== Final Weather-Enriched Dataset ===
Rows: 3,722,261 | Columns: 30


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,ride_id,member_casual,rideable_type,ride_length_seconds,ride_length_7d_avg_seconds,ride_length_mad_zscore,ride_length_outlier,dt_start_hour,weather_location_id,weather_latitude,weather_longitude,distance_miles,temperature_7ft_f,relative_humidity_7ft_pct,precipitation_in,rain_in,snowfall_in,wind_speed_33ft_mph,wind_direction_33ft_deg,cloud_cover_pct
0,2025-03-01 00:00:00,5,Woodlawn Ave & 55th St,TA1307000164,41.795275,-87.596475,Blackstone Ave & 59th St,22004,41.787875,-87.590475,96C3FBE4281EDBF6,member,classic_bike,240,240.000000,-0.755440,False,2025-03-01,0,41.65202,-87.78903,14.020366,35.78,63.0,0.0,0.0,0.0,10.501170,308.0,100.0
1,2025-03-01 00:00:00,5,Halsted St & Roscoe St,TA1309000025,41.943625,-87.649075,Sheffield Ave & Wrightwood Ave,TA1309000023,41.928725,-87.653825,C7F5B9031C0C0611,member,electric_bike,400,320.000000,-0.323760,False,2025-03-01,1,42.07381,-87.37610,16.652622,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
2,2025-03-01 00:05:00,5,Clark St & Chicago Ave,13303,41.896725,-87.630875,Wells St & Elm St,KA1504000135,41.903225,-87.634325,4AA5A994AE15E095,member,electric_bike,160,266.666667,-0.971280,False,2025-03-01,1,42.07381,-87.37610,17.914124,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
3,2025-03-01 00:05:00,5,Racine Ave & Belmont Ave,TA1308000019,41.939725,-87.658875,Clark St & Armitage Ave,13146,41.918325,-87.636275,28891BF75399F400,casual,electric_bike,490,490.000000,-0.460634,False,2025-03-01,1,42.07381,-87.37610,17.222020,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
4,2025-03-01 00:05:00,5,Wilton Ave & Belmont Ave,TA1307000134,41.940225,-87.652925,Clark St & Wrightwood Ave,TA1305000014,41.929525,-87.643125,FA66DD32E79E602A,casual,electric_bike,270,380.000000,-0.822561,False,2025-03-01,1,42.07381,-87.37610,16.946416,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
5,2025-03-01 00:05:00,5,Cornell Ave & Hyde Park Blvd,KA1503000007,41.802425,-87.586925,Shore Dr & 55th St,TA1308000009,41.795225,-87.580725,8DB8FFA1A678BAAE,member,classic_bike,210,252.500000,-0.836380,False,2025-03-01,0,41.65202,-87.78903,14.717511,35.78,63.0,0.0,0.0,0.0,10.501170,308.0,100.0
6,2025-03-01 00:05:00,5,Damen Ave & Grand Ave,TA1308000006,41.892375,-87.676875,Western Ave & Walton St,KA1504000103,41.898425,-87.686575,B40FAA4845607B43,casual,electric_bike,350,370.000000,-0.690951,False,2025-03-01,0,41.65202,-87.78903,17.583804,35.78,63.0,0.0,0.0,0.0,10.501170,308.0,100.0
7,2025-03-01 00:05:00,5,Dearborn Pkwy & Delaware Pl,TA1307000128,41.898975,-87.629925,Canal St & Madison St,13341,41.882425,-87.639775,8A27F4E7D27C870A,member,classic_bike,890,380.000000,0.998260,False,2025-03-01,1,42.07381,-87.37610,17.772320,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
8,2025-03-01 00:05:00,5,Clark St & Lunt Ave,KA1504000162,42.009025,-87.674125,Glenwood Ave & Morse Ave,KA1504000175,42.007975,-87.665525,ADDB2D3718363D1A,member,electric_bike,160,343.333333,-0.971280,False,2025-03-01,1,42.07381,-87.37610,15.934161,32.72,69.0,0.0,0.0,0.0,27.029639,326.0,100.0
9,2025-03-01 00:05:00,5,Clinton St & Lake St,13021,41.885625,-87.641825,Sangamon St & Lake St,TA1306000015,41.885775,-87.651025,A9B7058ACF1B24A3,casual,classic_bike,320,357.500000,-0.740305,False,2025-03-01,0,41.65202,-87.78903,17.834283,35.78,63.0,0.0,0.0,0.0,10.501170,308.0,100.0


### Export Final Enriched Data
This cell validates export prerequisites, writes the final weather-enriched dataset in the configured format, and then removes local cache directories used during data prep.

**Section-aligned script flow (matches the Python sections below)**
1. **Section 1 - Validate required variables and export format**
   - Confirms required objects exist: `df_weather_merge`, `start_yyyymm`, `end_yyyymm`, `OUTPUT_DIR`, `export_format`.
   - Validates `export_format` is one of: `csv`, `zip`.

2. **Section 2 - Build output filenames**
   - Constructs `base_filename` from the selected month range.
   - Builds output paths under `OUTPUT_DIR`.

3. **Section 3 - Define cache directories for cleanup**
   - Declares cleanup targets: `.divvy_zip`, `.divvy_csv`, `.weather_csv`.

4. **Section 4 - Export data**
   - `csv` mode: writes a single CSV file.
   - `zip` mode: writes a temporary CSV, packages it into a ZIP archive, then removes the temp file.
   - Prints row/column export counts and destination path.

5. **Section 5 - Cleanup cache directories**
   - Attempts to remove each cache directory after export.
   - Prints per-directory status: removed, skipped (not found), or error.

**Outputs from this cell**
- Exported enriched dataset file: `{start_yyyymm}-{end_yyyymm}-divvy-tripdata-enriched.csv` or `.zip`.
- Optional ZIP artifact containing one CSV (when `export_format='zip'`).
- Cleanup status logs for `.divvy_zip`, `.divvy_csv`, and `.weather_csv`.

In [40]:
# ================================================================================
# EXPORT FINAL ENRICHED DATA
# ================================================================================
# Purpose: export df_weather_merge to configured format and remove local cache directories.

# Export Final Enriched Data with cleanup (df_weather_merge only)
# ================================================================================
# SECTION 1: VALIDATE REQUIRED VARIABLES & EXPORT FORMAT
# ================================================================================
# Validate required variables and export format exist.
required_vars = ['df_weather_merge', 'start_yyyymm', 'end_yyyymm', 'OUTPUT_DIR', 'export_format']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(f"Missing required variables for export: {missing_vars}")

if export_format not in ['csv', 'zip']:
    raise ValueError(f"export_format must be 'csv' or 'zip', got: {export_format}")

# ================================================================================
# SECTION 2: BUILD OUTPUT FILENAMES
# ================================================================================
# Build output names from pre-existing date-range variables.
base_filename = f"{start_yyyymm}-{end_yyyymm}-divvy-tripdata-enriched"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_output_path = OUTPUT_DIR / f"{base_filename}.csv"

# ================================================================================
# SECTION 3: DEFINE CACHE DIRECTORIES FOR CLEANUP
# ================================================================================
# Validate required cache directories exist before cleanup.
cache_dirs = [zip_dir, extract_dir, weather_dir]

# ================================================================================
# SECTION 4: EXPORT DATA
# ================================================================================
# Export data in specified format.
if export_format == 'csv':
    # Export as CSV (single file)
    df_weather_merge.to_csv(csv_output_path, index=False)
    print("✓ Export format: CSV")
    print(f"Rows exported: {len(df_weather_merge):,}")
    print(f"Columns exported: {len(df_weather_merge.columns)}")
    print(f"CSV exported to: {csv_output_path}")
    
elif export_format == 'zip':
    # Export as ZIP archive (compressed single file)
    temp_csv_for_zip = OUTPUT_DIR / f".{base_filename}_temp.csv"
    df_weather_merge.to_csv(temp_csv_for_zip, index=False)
    
    zip_output_path = OUTPUT_DIR / f"{base_filename}.zip"
    shutil.make_archive(
        str(zip_output_path.with_suffix('')),
        'zip',
        OUTPUT_DIR,
        temp_csv_for_zip.name
    )
    temp_csv_for_zip.unlink()  # Remove temporary CSV
    
    print("✓ Export format: ZIP")
    print(f"Rows exported: {len(df_weather_merge):,}")
    print(f"Columns exported: {len(df_weather_merge.columns)}")
    print(f"ZIP archive exported to: {zip_output_path}")

# ================================================================================
# SECTION 5: CLEANUP CACHE DIRECTORIES
# ================================================================================
# CLEANUP: Remove cache directories to free disk space.
print("\n" + "="*60)
print("CLEANUP: Removing cache directories...")
print("="*60)

for cache_dir in cache_dirs:
    if cache_dir.exists():
        try:
            shutil.rmtree(cache_dir)
            print(f"✓ Removed: {cache_dir.name}")
        except Exception as e:
            print(f"✗ Error removing {cache_dir.name}: {e}")
    else:
        print(f"⊘ Not found (skipped): {cache_dir.name}")

print("\n✓ Export and cleanup complete!")
print(f"Final enriched dataset: {base_filename}.{export_format}")

✓ Export format: ZIP
Rows exported: 3,722,261
Columns exported: 30
ZIP archive exported to: /mnt/RepoRetLabs/code/jupyter-notebooks/case-study_bike-share-success/202503-202602-divvy-tripdata-enriched.zip

CLEANUP: Removing cache directories...
✓ Removed: .divvy_zip
✓ Removed: .divvy_csv
✓ Removed: .weather_csv

✓ Export and cleanup complete!
Final enriched dataset: 202503-202602-divvy-tripdata-enriched.zip
